In [ ]:
import importlib.metadata
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

def _ensure_transformers(min_ver=(5, 0, 0)):
    need_install = False
    try:
        ver_str = importlib.metadata.version("transformers").split("+")[0].split("a")[0].split("b")[0].split("rc")[0]
        parts = tuple(int(x) for x in ver_str.split(".")[:3])
        if parts < min_ver:
            need_install = True
    except Exception:
        need_install = True
    if need_install:
        print("[SETUP] Installing transformers>=5.0.0...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.0.0"], check=True)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    _ensure_transformers()

import numpy as np
import torch
import torchvision
import transformers

CHECKPOINT_DIR = Path("/content/checkpoints") if IN_COLAB else Path("./checkpoints")
ENSEMBLE_DIR = CHECKPOINT_DIR / "ensemble"
OUTPUT_DIR_S2 = Path("/content/stage2_results") if IN_COLAB else Path("./stage2_results")
ZIP_EXPORT_S2 = Path("/content/SE374_Stage2_Training_Results.zip") if IN_COLAB else Path("./SE374_Stage2_Training_Results.zip")
for _d in (CHECKPOINT_DIR, ENSEMBLE_DIR, OUTPUT_DIR_S2):
    _d.mkdir(parents=True, exist_ok=True)

LOCAL_BUNDLE = Path("/content/local_repo.bundle")
STAGE2_GIT_REF = os.environ.get("STAGE2_GIT_REF", "").strip()
if LOCAL_BUNDLE.is_file():
    _repo_dst = Path("/content/medical_wsss_repo")
    if _repo_dst.exists():
        shutil.rmtree(_repo_dst, ignore_errors=True)
    print(f"[SETUP] Cloning local git bundle {LOCAL_BUNDLE} -> {_repo_dst}", flush=True)
    subprocess.run(["git", "clone", str(LOCAL_BUNDLE), str(_repo_dst)], check=True)
    CODE_ROOT = str(_repo_dst.resolve())
    if STAGE2_GIT_REF:
        _br = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "--verify", "--quiet", f"refs/remotes/origin/{STAGE2_GIT_REF}^{{commit}}"], capture_output=True, text=True, check=False)
        if _br.returncode == 0:
            _co = subprocess.run(["git", "-C", CODE_ROOT, "checkout", "-B", STAGE2_GIT_REF, f"refs/remotes/origin/{STAGE2_GIT_REF}"], capture_output=True, text=True, check=False)
        else:
            _co = subprocess.run(["git", "-C", CODE_ROOT, "checkout", "--force", STAGE2_GIT_REF], capture_output=True, text=True, check=False)
        if _co.returncode != 0:
            raise RuntimeError(f"Failed to checkout STAGE2_GIT_REF={STAGE2_GIT_REF!r} from {LOCAL_BUNDLE}: {_co.stderr.strip()}")
else:
    CANDIDATES = [
        Path("/content/medical_wsss_repo"),
        Path("/content/medical-wsss"),
        Path.cwd(),
        Path.cwd().parent,
    ]
    CODE_ROOT = next((str(p.resolve()) for p in CANDIDATES if (p / "src" / "models").is_dir()), None)
    if CODE_ROOT is None:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/khoinguyen59/medical-wsss.git", "/content/medical_wsss_repo"], check=True)
        CODE_ROOT = str(Path("/content/medical_wsss_repo").resolve())
    if STAGE2_GIT_REF:
        if not (Path(CODE_ROOT) / ".git").exists():
            raise RuntimeError(f"STAGE2_GIT_REF={STAGE2_GIT_REF!r} requested, but {CODE_ROOT} is not a git repo")
        subprocess.run(["git", "-C", CODE_ROOT, "reset", "--hard", "HEAD"], capture_output=True, text=True, check=True)
        subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", "+refs/heads/*:refs/remotes/origin/*", "--tags"], capture_output=True, text=True, check=False)
        subprocess.run(["git", "-C", CODE_ROOT, "fetch", "--depth", "1", "origin", STAGE2_GIT_REF], capture_output=True, text=True, check=False)
        _br = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "--verify", "--quiet", f"refs/remotes/origin/{STAGE2_GIT_REF}^{{commit}}"], capture_output=True, text=True, check=False)
        if _br.returncode == 0:
            subprocess.run(["git", "-C", CODE_ROOT, "checkout", "-B", STAGE2_GIT_REF, f"refs/remotes/origin/{STAGE2_GIT_REF}"], capture_output=True, text=True, check=True)
            subprocess.run(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", STAGE2_GIT_REF], capture_output=True, text=True, check=True)
        else:
            if (Path(CODE_ROOT) / ".git" / "shallow").exists():
                subprocess.run(["git", "-C", CODE_ROOT, "fetch", "--unshallow", "origin"], capture_output=True, text=True, check=False)
            subprocess.run(["git", "-C", CODE_ROOT, "checkout", "--force", STAGE2_GIT_REF], capture_output=True, text=True, check=True)
    elif (Path(CODE_ROOT) / ".git").exists() and IN_COLAB:
        _hb = subprocess.run(["git", "-C", CODE_ROOT, "symbolic-ref", "--short", "-q", "HEAD"], capture_output=True, text=True, check=False)
        if _hb.returncode == 0 and _hb.stdout.strip():
            subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", _hb.stdout.strip()], capture_output=True, text=True, check=False)
            subprocess.run(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", _hb.stdout.strip()], capture_output=True, text=True, check=False)

for _k in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    sys.modules.pop(_k, None)
if CODE_ROOT in sys.path:
    sys.path.remove(CODE_ROOT)
sys.path.insert(0, CODE_ROOT)
os.environ["PYTHONPATH"] = CODE_ROOT + (os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")

SEED = int(os.environ.get("STAGE2_SEED", "42"))
DRY_RUN = os.environ.get("STAGE2_DRY_RUN", "0").strip().lower() in ("1", "true", "yes")
NUM_WORKERS = int(os.environ.get("STAGE2_NUM_WORKERS", "4"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (
    torch.bfloat16
    if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported(including_emulation=False))
    else torch.float16
)
print(f"[READY] Code={CODE_ROOT} | Mode=gate7_clinical_luad_bench | DryRun={DRY_RUN} | Seed={SEED}", flush=True)
print(f"[READY] Compute={DEVICE} | AMP={AMP_DTYPE} | torch={torch.__version__} | tv={torchvision.__version__} | tf={transformers.__version__}", flush=True)


In [ ]:
import urllib.request
import zipfile
from torch.utils.data import DataLoader
from src.colab_pkg.downloader import (
    GDRIVE_DATASET_ID,
    HF_DATASET_URL,
    ensure_luad_dataset,
    resolve_split_dirs,
)
from src.stage2_v2.gate7_clinical_bench import LUADBCSSAlignedDataset, extract_luad_case_id
from src.training.dataset_loader import BCSSStage2ValDataset, load_bcss_patient_splits

DATASET_DIR = Path("/content/BCSS-WSSS") if IN_COLAB else Path("./data/BCSS-WSSS")
DATASET_ZIP = Path("/content/BCSS-WSSS.zip") if IN_COLAB else Path("./data/BCSS-WSSS.zip")
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

_, TEST_IMG_DIR, TEST_MASK_DIR = resolve_split_dirs(DATASET_DIR, split="test", min_images=1, min_patients=1)
if TEST_IMG_DIR is None or TEST_MASK_DIR is None:
    if not DATASET_ZIP.is_file() or DATASET_ZIP.stat().st_size < 1_000_000:
        print(f"[DOWNLOAD] Fetching BCSS-WSSS from {HF_DATASET_URL}", flush=True)
        try:
            urllib.request.urlretrieve(HF_DATASET_URL, str(DATASET_ZIP))
        except Exception:
            subprocess.run(["pip", "install", "-q", "gdown"], check=False)
            subprocess.run(["gdown", "--id", GDRIVE_DATASET_ID, "-O", str(DATASET_ZIP)], check=True)
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(str(DATASET_DIR))
    _, TEST_IMG_DIR, TEST_MASK_DIR = resolve_split_dirs(DATASET_DIR, split="test", min_images=1, min_patients=1)

if TEST_IMG_DIR is None or TEST_MASK_DIR is None:
    raise RuntimeError(f"[STAGE2 ERROR] Missing BCSS test split directories under {DATASET_DIR}")

splits = load_bcss_patient_splits()
test_pids = splits["test"]
assert set(test_pids).isdisjoint(set(splits["train"]) | set(splits["val"])), "Patient leakage detected!"

sample_lim = 64 if DRY_RUN else None
bcss_test_ds = BCSSStage2ValDataset(
    TEST_IMG_DIR,
    TEST_MASK_DIR,
    patient_ids=test_pids,
    limit=sample_lim,
    strict_patient_filter=True,
    seed=SEED,
)

LUAD_DIR = Path("/content/LUAD-HistoSeg") if IN_COLAB else Path("./data/LUAD-HistoSeg")
LUAD_ZIP = Path("/content/LUAD-HistoSeg.zip") if IN_COLAB else Path("./data/LUAD-HistoSeg.zip")
_, LUAD_IMG_DIR, LUAD_MASK_DIR = ensure_luad_dataset(
    data_root=LUAD_DIR,
    zip_path=LUAD_ZIP,
    extract_dir=LUAD_DIR,
    drive_cache_dir=None,
    split="test",
)
luad_test_ds = LUADBCSSAlignedDataset(LUAD_IMG_DIR, LUAD_MASK_DIR, target_size=(224, 224), limit=sample_lim)
luad_pids = sorted({extract_luad_case_id(p.name) for p in luad_test_ds.files})

nw_eval = min(NUM_WORKERS, os.cpu_count() or 2)
dl_kw = {"prefetch_factor": 2, "multiprocessing_context": "spawn"} if nw_eval > 0 else {}
bcss_test_loader = DataLoader(bcss_test_ds, batch_size=16, shuffle=False, num_workers=nw_eval, pin_memory=(DEVICE.type == "cuda"), **dl_kw)
luad_test_loader = DataLoader(luad_test_ds, batch_size=16, shuffle=False, num_workers=nw_eval, pin_memory=(DEVICE.type == "cuda"), **dl_kw)
print(f"[DATASET] BCSS Test={len(bcss_test_ds)} patches ({len(test_pids)} disjoint patients)", flush=True)
print(f"[DATASET] LUAD Test={len(luad_test_ds)} patches ({len(luad_pids)} distinct cases)", flush=True)


In [ ]:
from src.colab_pkg.downloader import HF_STAGE2_URL
from src.models.wsss_inference_methods import safe_torch_load
from src.stage2_v2.ensemble import (
    DEFAULT_ARM_FAMILIES,
    SHA256_AUTHOR_STAGE2,
    SHA256_STAGE1_BEST_CLS,
    SHA256_STAGE1_RETRAINED,
    validate_track_a_arm_spec,
)
from src.stage2_v2.visual_audit import GATE7_ARM_ORDER, GATE7_ARM_SHA256
from src.training.eval_stage1_full import compute_file_sha256

def _find_ckpt(candidates, expected_sha=None, download_url=None, label="ckpt"):
    existing = [p for p in candidates if p.is_file()]
    path = None
    if expected_sha:
        for cand in existing:
            if compute_file_sha256(cand) == expected_sha:
                path = cand
                break
        if path is None and existing and not download_url:
            bad_sha = compute_file_sha256(existing[0])
            raise RuntimeError(f"[STAGE2 ERROR] SHA256 mismatch for {label} ({existing[0]}): got {bad_sha}, expected {expected_sha}")
    elif existing:
        path = existing[0]
    if path is None and download_url:
        path = candidates[0]
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"[DOWNLOAD] Fetching {label} -> {path}", flush=True)
        urllib.request.urlretrieve(download_url, str(path))
    if path is None or not path.is_file():
        raise RuntimeError(f"[STAGE2 ERROR] Missing {label}: checked {[str(c) for c in candidates]}")
    sha = compute_file_sha256(path)
    if expected_sha and sha != expected_sha:
        raise RuntimeError(f"[STAGE2 ERROR] SHA256 mismatch for {label} ({path}): got {sha}, expected {expected_sha}")
    print(f"[CKPT OK] {label}: {path} (sha256={sha[:12]})", flush=True)
    return path

STAGE2_TRACK = os.environ.get("STAGE2_TRACK", "track_a_clean").strip().lower()
_raw_arms = os.environ.get("STAGE2_ENSEMBLE_ARMS", "").strip()
ENSEMBLE_ARMS = [a.strip().lower() for a in _raw_arms.split(",") if a.strip()] if _raw_arms else list(GATE7_ARM_ORDER)
if len(ENSEMBLE_ARMS) != 4 or len(set(ENSEMBLE_ARMS)) != 4:
    raise ValueError(f"[STAGE2 ERROR] STAGE2_ENSEMBLE_ARMS must specify 4 distinct arm names, got: {ENSEMBLE_ARMS}")
print(f"[BENCH CONFIG] Track={STAGE2_TRACK} | Arms={ENSEMBLE_ARMS}", flush=True)

arm_ckpt_map = {}
vps_ev_dirs = [CHECKPOINT_DIR / "vps_evidence", Path(CODE_ROOT) / "checkpoints" / "vps_evidence"]
search_dirs = [ENSEMBLE_DIR, CHECKPOINT_DIR, Path(CODE_ROOT) / "checkpoints"]
ensemble_pth_files = sorted(ENSEMBLE_DIR.glob("*.pth")) if ENSEMBLE_DIR.is_dir() else []
for arm_name in ENSEMBLE_ARMS:
    short_arm = arm_name.split("_")[0]
    exp_sha = GATE7_ARM_SHA256.get(arm_name)
    cands = []
    for d in search_dirs:
        cands.extend([d / f"{arm_name}.pth", d / f"{short_arm}.pth", d / f"{short_arm}_best.pth"])
    for ev_d in vps_ev_dirs:
        if ev_d.is_dir():
            cands.extend(sorted(ev_d.glob(f"*_g7_{short_arm}_*/stage2_retrained_best.pth")))
    existing_cands = [c for c in cands if c.is_file()]
    hit = None
    if exp_sha:
        hit = next((c for c in existing_cands if compute_file_sha256(c) == exp_sha), None)
        if hit is None:
            hit = next((p for p in ensemble_pth_files if compute_file_sha256(p) == exp_sha), None)
    if hit is None and existing_cands:
        hit = existing_cands[0]
    if hit is not None:
        arm_ckpt_map[arm_name] = hit

if len(arm_ckpt_map) < 4:
    for pth in ensemble_pth_files:
        if pth in arm_ckpt_map.values():
            continue
        try:
            meta = safe_torch_load(pth, device=torch.device("cpu"))
            preset = str(meta.get("preset", "")).strip().lower() if isinstance(meta, dict) else ""
            if preset in ENSEMBLE_ARMS and preset not in arm_ckpt_map:
                arm_ckpt_map[preset] = pth
        except Exception:
            pass

missing_arms = [a for a in ENSEMBLE_ARMS if a not in arm_ckpt_map]
if missing_arms:
    raise RuntimeError(f"[STAGE2 ERROR] Missing Stage-2 arm checkpoints for: {missing_arms} in {ENSEMBLE_DIR}")

arm_specs = []
for arm_name in ENSEMBLE_ARMS:
    ckpt_p = arm_ckpt_map[arm_name]
    validate_track_a_arm_spec({"name": arm_name, "ckpt_path": ckpt_p}, track=STAGE2_TRACK)
    sha = compute_file_sha256(ckpt_p)
    exp_sha = GATE7_ARM_SHA256.get(arm_name)
    if exp_sha and sha != exp_sha:
        raise RuntimeError(f"[STAGE2 ERROR] SHA256 mismatch for {arm_name}: got {sha}, expected {exp_sha}")
    print(f"[ARM OK] {arm_name}: {ckpt_p.name} (sha256={sha[:12]})", flush=True)
    arm_specs.append({"name": arm_name, "family": DEFAULT_ARM_FAMILIES.get(arm_name, "unknown"), "ckpt_path": ckpt_p})

s1_cands = [
    ENSEMBLE_DIR / "stage1_best_cls.pth",
    CHECKPOINT_DIR / "stage1_best_cls.pth",
    Path(CODE_ROOT) / "checkpoints" / "stage1_best_cls.pth",
]
s1_ret_cands = [
    ENSEMBLE_DIR / "stage1_retrained_best.pth",
    CHECKPOINT_DIR / "vps_gate4_winner" / "stage1_retrained_best.pth",
    Path(CODE_ROOT) / "checkpoints" / "vps_gate4_winner" / "stage1_retrained_best.pth",
    CHECKPOINT_DIR / "stage1_retrained_best.pth",
    Path(CODE_ROOT) / "checkpoints" / "stage1_retrained_best.pth",
]
for ev_d in vps_ev_dirs:
    if ev_d.is_dir():
        s1_cands.extend(sorted(ev_d.glob("*_arm_b_full/checkpoints/stage1_best_cls.pth")))
        s1_ret_cands.extend(sorted(ev_d.glob("*_arm_b_full/checkpoints/stage1_retrained_best.pth")))

STAGE1_BEST_CLS_PATH = _find_ckpt(s1_cands, expected_sha=SHA256_STAGE1_BEST_CLS, label="stage1_best_cls.pth")
STAGE1_RETRAINED_PATH = _find_ckpt(s1_ret_cands, expected_sha=SHA256_STAGE1_RETRAINED, label="stage1_retrained_best.pth")
AUTHOR_STAGE2_PATH = _find_ckpt(
    [
        ENSEMBLE_DIR / "stage2_checkpoint_trained_on_bcss.pth",
        CHECKPOINT_DIR / "stage2_checkpoint_trained_on_bcss.pth",
        Path(CODE_ROOT) / "checkpoints" / "stage2_checkpoint_trained_on_bcss.pth",
    ],
    expected_sha=SHA256_AUTHOR_STAGE2,
    download_url=HF_STAGE2_URL,
    label="stage2_checkpoint_trained_on_bcss.pth",
)


In [ ]:
from src.stage2_v2.gate7_clinical_bench import run_gate7_clinical_and_luad_benchmark

results = run_gate7_clinical_and_luad_benchmark(
    arm_specs=arm_specs,
    bcss_test_loader=bcss_test_loader,
    luad_test_loader=luad_test_loader,
    stage1_cls_ckpt=STAGE1_BEST_CLS_PATH,
    author_stage1_ckpt=STAGE1_RETRAINED_PATH,
    author_stage2_ckpt=AUTHOR_STAGE2_PATH,
    output_dir=OUTPUT_DIR_S2,
    zip_path=ZIP_EXPORT_S2,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    track=STAGE2_TRACK,
    n_warmup=3 if DRY_RUN else 5,
    n_iters=5 if DRY_RUN else 20,
    dpi=300,
)
print("[GATE7 CLINICAL & LUAD BENCHMARK COMPLETE]", flush=True)


In [ ]:
print("=== GATE-7 CLINICAL BIOMARKERS, SURVIVAL, GPU EFFICIENCY & LUAD SUMMARY ===", flush=True)
print("BCSS TEST Pooled mIoU (%):", results["bcss_pooled_miou_pct"], flush=True)
for mk, mst in results["clinical_biomarkers"]["models"].items():
    print(
        f"  [BIOMARKERS {mk}] VTP r={mst['vtp']['pearson_r']:.4f} (p={mst['vtp']['pearson_p']:.2e}, bias={mst['vtp']['mean_bias']:+.2f}%) | "
        f"TSR r={mst['tsr']['pearson_r']:.4f} | CAP acc={mst['cap_concordance']['accuracy']:.4f} kappa={mst['cap_concordance']['cohens_kappa']:.4f} | "
        f"KM p={mst['km_logrank_p']:.4f}",
        flush=True,
    )
for mk, gst in results["gpu_efficiency"]["models"].items():
    print(
        f"  [GPU {mk}] {gst['latency_ms_per_patch']:.2f} ms/patch | {gst['throughput_fps']:.1f} FPS | "
        f"VRAM={gst['peak_vram_mb']:.1f} MB | Params={gst['param_count_m']:.1f}M",
        flush=True,
    )
if results.get("luad_cross_organ"):
    luad = results["luad_cross_organ"]
    print(f"  [LUAD ZERO-SHOT] n_patches={luad['n_patches']} n_patients={luad['n_patients']}", flush=True)
    for mk, lm in luad["models"].items():
        print(f"    {mk}: mIoU={lm['mIoU']:.4f} mDice={lm['mDice']:.4f} per_class={lm['per_class_iou']}", flush=True)


In [ ]:
import zipfile

required_files = (
    "STAGE2_TRAINING_METRICS.json",
    "GATE7_CLINICAL_BENCH_METRICS.json",
    "patient_clinical_risk_summary.csv",
    "gate7_clinical_biomarkers.json",
    "bland_altman_tsr_vtp_bw.png",
    "km_survival_tsr_bw.png",
    "gate7_gpu_efficiency.json",
    "gpu_pareto_efficiency_bw.png",
    "gate7_luad_cross_organ.json",
    "stage2_training_loss.png",
)
for fname in required_files:
    if not (OUTPUT_DIR_S2 / fname).is_file():
        raise RuntimeError(f"[STAGE2 ERROR] Missing expected deliverable: {OUTPUT_DIR_S2 / fname}")
if not ZIP_EXPORT_S2.is_file():
    raise RuntimeError(f"[STAGE2 ERROR] Missing ZIP export: {ZIP_EXPORT_S2}")

with zipfile.ZipFile(ZIP_EXPORT_S2, "r") as zf:
    names = set(zf.namelist())
    for fname in required_files:
        if fname not in names:
            raise RuntimeError(f"[STAGE2 ERROR] {fname} missing from ZIP bundle.")

zip_mb = ZIP_EXPORT_S2.stat().st_size / (1024.0 * 1024.0)
print(f"[EXPORT OK] ZIP={ZIP_EXPORT_S2} ({zip_mb:.2f} MB) | Files={len(required_files)}", flush=True)
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(ZIP_EXPORT_S2))
    except Exception as exc:
        print(f"[NOTICE] Colab browser download skipped: {exc}", flush=True)
